In [0]:
# ── Configuration ────────────────────────────────────────────────────────────
# All tuneable parameters live here so nothing is hard-coded deeper in the notebook.

# Date-range presets for data loading
DATE_RANGES = {
    # "full": {"start": "2022-06-30", "end": "2024-07-01", "mob_months": 25},
     "full": {"start": "2022-06-30", "end": "2025-07-01", "mob_months": 13},
    "recent": {"start": "2025-01-31", "end": "2026-07-01", "mob_months": 37},
}

# Bad-definition candidates (min_bucket follows the DELINQUENCYBUCKET encoding:
#   0=current, 1=1-29 DPD, 2=30-59 DPD, 3=60-89 DPD, 4=90+ DPD)
BAD_DEFINITIONS = {
    "Ever60DPD_12M": {"min_bucket": 3, "window_months": 12},
    "Ever30DPD_6M":  {"min_bucket": 2, "window_months": 6},
    "Ever90DPD_12M": {"min_bucket": 4, "window_months": 12},
}

# Target generation: 12-month performance window
TARGET_WINDOW_MONTHS = 12

# Minimum OPEN snapshots a card must have to be retained
MIN_OPEN_SNAPSHOTS = 3

# Columns needed downstream (avoids OOM when converting full 88-col frame to pandas)
ANALYSIS_COLS = [
    "SNAPSHOT_DATE", "CRNNO", "CIF", "CREDITCARDNO", "APPROVEDATE",
    "DELINQUENCYBUCKET", "CURRENT_STATUS", "REFERENCE_NUMBER", "FTS_Ref_No","GENERALSTAT"
]

# Output directory for charts
OUTPUT_DIR = "/Volumes/mis_automation/datascience_pbg/pbg_cc_ascore/input/"

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col

In [0]:
def load_and_clean_credit_card_data(date_start, date_end, mob_months=37):
    """
    Load credit card history and apply business exclusion filters.

    Parameters
    ----------
    date_start : str   – APPROVEDATE lower bound (exclusive), e.g. '2022-06-30'
    date_end   : str   – APPROVEDATE upper bound (exclusive), e.g. '2025-07-01'
    mob_months : int   – months-on-book observation window (default 37)

    Returns
    -------
    pyspark.sql.DataFrame  – cleaned Spark DataFrame with exclusions applied.
    """
    # Step 1: Load raw data within the observation window
    df = spark.sql(f"""
        SELECT *
        FROM mis_automation.silver_credit_card.vw_credit_card_history
        WHERE SNAPSHOT_DATE >= APPROVEDATE
          AND SNAPSHOT_DATE < add_months(APPROVEDATE, {mob_months})
          AND APPROVEDATE > '{date_start}'
          AND APPROVEDATE < '{date_end}'
          AND UPPER(TRIM(CARD_BUSINESS_SEGMENT)) IN ('CARDS')
          AND UPPER(TRIM(CUSTOMER_SEGMENT)) = 'SALARIED'
    """)

    # Step 2: Count OPEN snapshots per card (needed for Exclusion 9)
    w = Window.partitionBy("CREDITCARDNO")
    df = df.withColumn(
        "open_count",
        F.sum(F.when(F.col("CURRENT_STATUS") == "1.OPEN", 1).otherwise(0)).over(w),
    )

    # Step 3: Exclusion conditions
    exclude_cond = (
        (F.col("VIP") == "1")                                                             # 3a: VIP customers
        | (F.upper(F.trim(F.col("RULING_FAMILY_FLAG"))) == "Y")                             # 3b: Ruling family
        | (F.col("PRIMARYCARD") == 0)                                                      # 7: Supplementary cards
        # | (F.col("open_count") < 3)                                                        # 9: Fewer than 3 OPEN
        | (F.upper(F.trim(F.col("F_SEGMENT_NEW"))) == "SALARIED-IM")                        # 10a: Salaried-IM
        | (F.upper(F.trim(F.col("SEGMENT_POLICY"))).isin(
              "PL CUSTOMERS", "AUTO-LOAN CUSTOMERS", "MORTGAGE CXMS", "STAFF")))          # 10b: Unwanted segment policies
        # |(F.upper(F.trim(F.col("GENERALSTAT"))).isin('FRUD')))
        # |(F.upper(F.trim(F.col("GENERALSTAT"))).isin('SKIP', 'SKP7', 'SKIP', 'SPSK', 'FRUD')))

    inclusive_cond = (
        (F.col("CLASSCODE").isNull()| ~F.trim(F.col("CLASSCODE")).isin("STAFF", "SME"))   # 1, 8: SME/STAFF class code
        & (F.upper(F.trim(F.col("CARD_BUSINESS_SEGMENT"))).isin("CARDS"))                  # 2: CARDS segment
        & (F.upper(F.trim(F.col("CUSTOMER_SEGMENT"))) == "SALARIED")                       # 6: Salaried
    )        

    df_clean = df.filter(~exclude_cond).filter(inclusive_cond)#.drop("open_count")

    # Step 4: Keep only "active" customers — at least 4 out of 12 snapshot-months
    #         with CUROUTBALTOTAL < 0 (i.e. carrying a balance, indicating usage).
    # w_card = Window.partitionBy("CREDITCARDNO")
    # df_clean = (
    #     df_clean
    #     .withColumn(
    #         "_active_months",
    #         F.sum(
    #             F.when(F.col("CUROUTBALTOTAL") < 0, 1).otherwise(0)
    #         ).over(w_card),
    #     )
    #     .filter(F.col("_active_months") >= 4)
    #     .drop("_active_months")
    # )

    return df_clean


def generate_target(df, window_months=TARGET_WINDOW_MONTHS):
    """
    Assign one target label per CREDITCARDNO based on the max
    DELINQUENCYBUCKET within the first `window_months` after approval.

    Target mapping
    --------------
    -2 : immature (fewer than `window_months` unique snapshot months)
     0 : good    (max bucket == 0)
    -1 : indeterminate (max bucket in 1-3)
     1 : bad     (max bucket >= 4, i.e. 90+ DPD)
    """
    df = df.copy()
    df["SNAPSHOT_DATE"] = pd.to_datetime(df["SNAPSHOT_DATE"])
    df["APPROVEDATE"]   = pd.to_datetime(df["APPROVEDATE"])

    mask = (
        (df["SNAPSHOT_DATE"] >= df["APPROVEDATE"])
        & (df["SNAPSHOT_DATE"] < df["APPROVEDATE"] + pd.DateOffset(months=window_months))
    )
    window_df = df.loc[mask].copy()
    window_df["_snap_month"] = window_df["SNAPSHOT_DATE"].dt.to_period("M")

    agg = (
        window_df
        .groupby("CREDITCARDNO")
        .agg(max_bucket=("DELINQUENCYBUCKET", "max"),
             unique_months=("_snap_month", "nunique"))
        .reset_index()
    )

    conditions = [
        agg["unique_months"] < window_months,  # -2: immature
        agg["max_bucket"] >= 4,                 #  1: bad 3 is used
        agg["max_bucket"] == 0,                 #  0: good
        agg["max_bucket"].between(1, 3),        # -1: indeterminate
    ]
    agg["target"] = np.select(conditions, [-2, 1, 0, -1], default=np.nan)

    if agg["target"].isna().any():
        n_drop = agg["target"].isna().sum()
        print(f"WARNING: {n_drop} rows matched no target condition — dropped.")
        agg = agg.dropna(subset=["target"])
    agg["target"] = agg["target"].astype(int)

    id_cols = ["CRNNO", "CIF", "CREDITCARDNO", "APPROVEDATE", "APPROVEDATE",
               "REFERENCE_NUMBER", "FTS_Ref_No","GENERALSTAT"]
    id_df = df[id_cols].drop_duplicates(subset="CREDITCARDNO")
    result = id_df.merge(agg[["CREDITCARDNO", "target"]], on="CREDITCARDNO", how="inner")
    result["REFERENCE_NUMBER"] = (
        result["REFERENCE_NUMBER"].astype(str).str.replace(r'\.0$', '', regex=True)
    )
    return result


def load_aecb_feature_store(df_target):
    """
    Load the AECB feature-store table, filter to reference numbers
    present in `df_target`, merge, and split into thin-file subset.

    Optimised: uses a broadcast Spark join instead of .isin() with a
    Python list, which pushes the filter into the scan and avoids
    serialising thousands of reference numbers to every executor.

    Returns
    -------
    aecb_merged : pd.DataFrame – AECB features joined with target
    df_thin     : pd.DataFrame – subset where file_type == 'Thin File'
    """
    # ── 1. Build a small Spark DF of unique reference numbers ────────────
    ref_pd = (
        df_target[["REFERENCE_NUMBER"]]
        .drop_duplicates()
        .rename(columns={"REFERENCE_NUMBER": "ref_num"})
    )
    ref_pd["ref_num"] = ref_pd["ref_num"].astype(str)
    ref_spark = spark.createDataFrame(ref_pd)
    ref_spark.createOrReplaceTempView("_aecb_ref_nums")

    # ── 2. Push filter into Spark via broadcast join ────────────────────
    aecb_filtered = spark.sql("""
        SELECT /*+ BROADCAST(r) */ a.*
        FROM mis_automation.silver_bureauone.slv_credit_card_aecb_feature_store a
        INNER JOIN _aecb_ref_nums r
               ON CAST(a.Reference_Number AS STRING) = r.ref_num
    """)
    aecb_pd = aecb_filtered.toPandas()

    # ── 3. Pandas merge with target columns ─────────────────────────────
    aecb_pd["Reference_Number"] = aecb_pd["Reference_Number"].astype(str)
    df_target["REFERENCE_NUMBER"] = df_target["REFERENCE_NUMBER"].astype(str)

    aecb_merged = aecb_pd.merge(
        df_target[["REFERENCE_NUMBER", "target", "CREDITCARDNO", "CRNNO","CIF","GENERALSTAT","GENERALSTAT"]].drop_duplicates(),
        left_on="Reference_Number",
        right_on="REFERENCE_NUMBER",
        how="inner",
    )
    df_thin = aecb_merged.loc[aecb_merged["file_type_new"] == "Thin File"].reset_index(drop=True)

    print(f"AECB shape   : {aecb_pd.shape}")
    print(f"Merged shape : {aecb_merged.shape}")
    print(f"Thin-file    : {df_thin.shape[0]:,} rows  |  {df_thin['CREDITCARDNO'].nunique():,} cards")
    print(f"\nTarget distribution:\n{aecb_merged['target'].value_counts(dropna=False).sort_index()}")
    return aecb_merged, df_thin


def load_a_score(df_target):
    """
    Load A-Score data from the adhoc score table, filter to CRNNOs
    present in df_target, and merge.

    Returns the merged DataFrame with A-Score columns joined on CRNNO.
    """
    a_score = spark.sql("""
        SELECT * FROM mis_automation.silver_credit_card.cc_a_score_adhoc_data_file
    """)

    crnno_list = [str(c) for c in df_target["CRNNO"].unique().tolist()]
    a_score_filtered = (
        a_score.withColumn("CRNNO", F.col("CRNNO").cast("string"))
               .filter(F.col("CRNNO").isin(crnno_list))
    )
    a_score_pd = a_score_filtered.toPandas()
    a_score_pd["CRNNO"] = a_score_pd["CRNNO"].astype(str)
    df_target["CRNNO"] = df_target["CRNNO"].astype(str)

    a_score_merged = a_score_pd.merge(
        df_target,
        on="CRNNO",
        how="right",
    )

    print(f"A-Score raw shape      : {a_score_pd.shape}")
    print(f"A-Score merged shape   : {a_score_merged.shape}")
    print(f"Unique CRNNOs matched  : {a_score_merged['CRNNO'].nunique():,}")
    print(f"Columns: {list(a_score_pd.columns)}")
    return a_score_merged

In [0]:
cfg = DATE_RANGES["full"]
df_clean = load_and_clean_credit_card_data(cfg["start"], cfg["end"], cfg["mob_months"])
df_pd = df_clean.select(ANALYSIS_COLS).toPandas()

In [0]:
df_target = generate_target(df_pd)
print(f"Shape: {df_target.shape}")
print(f"\nTarget distribution:\n{df_target['target'].value_counts().sort_index()}")
df_target.head()

In [0]:
df_target.head()

In [0]:
# ── Call the optimised AECB loader ───────────────────────────────────────────
df_target_aecb, df_thin = load_aecb_feature_store(df_target)
df_target_aecb.columns

In [0]:
analysis_cols =['Reference_Number',
 'CreatedON',
 'AECB_Score',
 'score_missing_flag',
 'Rating',
  'REFERENCE_NUMBER',
 'target',
 'CREDITCARDNO',
 'CRNNO',
 'CIF',
 'GENERALSTAT',
 'file_type_new',
 'file_type']

In [0]:
df = df_target_aecb[analysis_cols]
df.head()

In [0]:
# ── Compare file_type vs file_type_new for analysis suitability ──────────────

print("=" * 80)
print("1. VALUE COUNTS")
print("=" * 80)
print("\n--- file_type ---")
print(df["file_type"].value_counts(dropna=False))
print("\n--- file_type_new ---")
print(df["file_type_new"].value_counts(dropna=False))

print("\n" + "=" * 80)
print("2. TARGET DISTRIBUTION BY SEGMENT")
print("=" * 80)
print("\n--- file_type × target ---")
ft_target = pd.crosstab(df["file_type"], df["target"], margins=True)
print(ft_target)
print("\n--- file_type × target (normalised %) ---")
print(pd.crosstab(df["file_type"], df["target"], normalize="index").round(4) * 100)

print("\n--- file_type_new × target ---")
ftn_target = pd.crosstab(df["file_type_new"], df["target"], margins=True)
print(ftn_target)
print("\n--- file_type_new × target (normalised %) ---")
print(pd.crosstab(df["file_type_new"], df["target"], normalize="index").round(4) * 100)

print("\n" + "=" * 80)
print("3. AECB_SCORE DISTRIBUTION BY SEGMENT")
print("=" * 80)

# Convert AECB_Score to numeric (coerce errors to NaN)
df["AECB_Score_num"] = pd.to_numeric(df["AECB_Score"], errors="coerce")

print("\n--- AECB_Score stats by file_type ---")
print(df.groupby("file_type")["AECB_Score_num"].describe().round(2))

print("\n--- AECB_Score stats by file_type_new ---")
print(df.groupby("file_type_new")["AECB_Score_num"].describe().round(2))

print("\n" + "=" * 80)
print("4. AECB_SCORE STD DEV & COEFFICIENT OF VARIATION")
print("=" * 80)

for col_name in ["file_type", "file_type_new"]:
    stats = (
        df.groupby(col_name)["AECB_Score_num"]
        .agg(["count", "mean", "std", "median", "min", "max"])
        .round(2)
    )
    stats["cv_%"] = ((stats["std"] / stats["mean"]) * 100).round(2)
    stats["non_null_pct"] = (
        (df.groupby(col_name)["AECB_Score_num"].apply(lambda x: x.notna().sum()) / 
         df.groupby(col_name)["AECB_Score_num"].size() * 100).round(2)
    ).values
    print(f"\n--- {col_name} ---")
    print(stats)

print("\n" + "=" * 80)
print("5. TARGET vs AECB_SCORE BY SEGMENT (mean score per target per segment)")
print("=" * 80)

for col_name in ["file_type", "file_type_new"]:
    pivot = (
        df.groupby([col_name, "target"])["AECB_Score_num"]
        .agg(["mean", "std", "count"])
        .round(2)
    )
    print(f"\n--- {col_name} × target → AECB_Score ---")
    print(pivot)

# ── Visual comparison ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for i, col_name in enumerate(["file_type", "file_type_new"]):
    # Box plot: AECB_Score by segment
    df.boxplot(column="AECB_Score_num", by=col_name, ax=axes[0, i])
    axes[0, i].set_title(f"AECB_Score by {col_name}")
    axes[0, i].set_xlabel(col_name)
    axes[0, i].set_ylabel("AECB_Score")

    # Stacked bar: target distribution by segment
    ct = pd.crosstab(df[col_name], df["target"], normalize="index")
    ct.plot(kind="bar", stacked=True, ax=axes[1, i], colormap="RdYlGn")
    axes[1, i].set_title(f"Target distribution by {col_name}")
    axes[1, i].set_ylabel("Proportion")
    axes[1, i].legend(title="target", bbox_to_anchor=(1.0, 1.0))
    axes[1, i].tick_params(axis="x", rotation=45)

plt.suptitle("file_type vs file_type_new — Comparison", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# ── Summary recommendation ──────────────────────────────────────────────────
print("\n" + "=" * 80)
print("6. GRANULARITY SUMMARY")
print("=" * 80)
print(f"\nfile_type     unique values: {df['file_type'].nunique()}  → {df['file_type'].unique().tolist()}")
print(f"file_type_new unique values: {df['file_type_new'].nunique()}  → {df['file_type_new'].unique().tolist()}")
print(f"\nAgreement between columns: {(df['file_type'] == df['file_type_new']).mean() * 100:.2f}% of rows match")

# Clean up helper column
df.drop(columns=["AECB_Score_num"], inplace=True)